# NeuroGolf-2026 — Bug-Free Blend [LB 5126]

**Public LB: 5126.37** (updated 2026-04-27, +10.14 from previous)

This is the **bug-free / clean** companion to our other public submission. It deliberately avoids the scoring-related profiler quirks discussed in the community (Constant-not-counted, dynamic-shape under-counting, Compress-with-fake-zero-batch, etc.) so that the score here reflects what the model would receive after the host's announced rescoring fixes (April 21st update).

## 🆕 Latest update — task363 silent-fail fix (+10.14 LB)

`task363` was a **silent-fail** task: the previous public ONNX (afr1ste 5395 lineage) passed `train + test + arc-gen` locally but contributed **0 pts** on Kaggle's hidden test. We replaced it with a 265/265 verified custom ONNX based on a 5-AI rule consultation (GPT/Gemini/DeepSeek/GLM/Kimi) over the full 261 arc-gen examples.

The winning rule (from GPT) was a **shift-based correlation** with a "sparse line tie-break": for `1x4` / `4x1` line seeds, stamp all length-4 `0/2` runs, but if exactly two candidates exist with exactly one on the same row/column as the seed, keep only that one (catches `train[1]` outlier without breaking arc-gen). Combined with a `3x3` diamond branch (center=5 + row-major greedy via single 5×5 Conv kernel) and a generic 2D `H1` sliding-template branch, the predictor passes 265/265 locally and gains **+10.14 LB on both the bug-free and bug-using tracks** — confirming the silent fail and that the hidden test agrees with our rule.

## 🎯 Two-track strategy

| Track | LB | Notebook |
|---|---:|---|
| **Bug-free (this notebook)** | **5126.37** | [konbu17/neurogolf-2026-bug-free-blend-lb-5126](https://www.kaggle.com/code/konbu17/neurogolf-2026-bug-free-blend-lb-5126) |
| Bug-tolerant (max LB) | 6254.65 | [konbu17/neurogolf-2026-blended-401-tasks-lb-6254](https://www.kaggle.com/code/konbu17/neurogolf-2026-blended-401-tasks-lb-6254) |

If the grader is updated to count Constant-op tensors, validate static shapes, and run with non-zero example inputs (the three planned changes mentioned in [discussion #692827](https://www.kaggle.com/competitions/neurogolf-2026/discussion/692827)), we expect the bug-tolerant submission to drop substantially while this one stays close to its current value.

## 📊 LB evolution

| Stage | LB |
|---|---:|
| v1 (initial release, 341 tasks) | 4215.85 |
| v11 (354 tasks + cheap rewrites + dim-scrub) | 4516.57 |
| v13 (395 tasks, V7-base + dim-scrub) | 4995.13 |
| v_clean_v1 (above + custom `task322` fix + 48 cheaper-clean swaps from our 401-task NB) | 5116.23 |
| **+ custom `task363` silent-fail fix (this update)** | **5126.37 (+10.14)** |

Total LB lift over v1: **+910 pts**.

## 🙏 Acknowledgements & Attribution

This notebook builds directly on the open NeuroGolf community. Sincere thanks to:

- **[@aliafzal9323](https://www.kaggle.com/aliafzal9323)** — original 286-task baseline ONNX collection that anchors every public blend.
- **[@artemnazemtsev](https://www.kaggle.com/artemnazemtsev)** — [4180-neurogolf-logic-driven-ensembling](https://www.kaggle.com/code/artemnazemtsev/4180-neurogolf-logic-driven-ensembling) and [neurogolf-logic-driven-ensembling-part-3](https://www.kaggle.com/code/artemnazemtsev/neurogolf-logic-driven-ensembling-part-3) — the logic-driven ensembling backbone, including part-3 which contributed many of the ~40 new tasks visible in our v84.
- **[@needless090](https://www.kaggle.com/needless090)** — [neurogolf-4250](https://www.kaggle.com/code/needless090/neurogolf-4250) — task coverage that pushed the public floor toward 4250 and beyond (the v5 release was a major contributor to V7's gain).
- **[@jonathanchan](https://www.kaggle.com/jonathanchan)** — [ngc26-constraint-smart-logic-mix-blending](https://www.kaggle.com/code/jonathanchan/ngc26-constraint-smart-logic-mix-blending) — the **394-task V7 base** that this notebook directly extends. The version chain V1→V7 made the latest improvements available within hours of upstream changes.
- **[@poonszesen](https://www.kaggle.com/poonszesen)** — [strategies-explained-openonnxscience](https://www.kaggle.com/code/poonszesen/strategies-explained-openonnxscience) — the **`dim_value`/`dim_param` scrubbing** technique we apply 53 times in this submission (47 tasks via our earlier v18 bisection + 6 new ones discovered on the V7 base).
- **[@hengck23](https://www.kaggle.com/hengck23)** — [neurogolf-60task-np-solution](https://www.kaggle.com/code/hengck23/neurogolf-60task-np-solution) and the accompanying MIT-licensed [50-task NumPy solution dataset](https://www.kaggle.com/datasets/hengck23/neurogolf-50task-np-solution) — invaluable as ground-truth verification for rule-based ONNX rewrites (we used this to validate t073 and other cheap-rewrite candidates).

Without these openly-shared notebooks, datasets, and submissions, single-person progress in this competition would be impractical.

The contributions added in *this* notebook (in addition to remixing the above) are:
1. Three hand-coded **cheap-rewrite ONNXs** (t389, t282, t015) — Python sources bundled.
2. A **53-task safe-scrub list** + a **12-task confirmed-BAD list** — the result of 50+ bisection submits across v18 and V7 bases, distinguishing scrubs that survive Kaggle's grader from those that don't.

## Step 0 — Locate the bundled v84 ONNXs

In [ ]:
import os
import zipfile

# The bundled dataset ships a single submission zip. Kaggle may either keep the zip
# intact under /kaggle/input/.../submission.zip or pre-extract it next to the
# task ONNXs — accept whichever form is present.
ZIP_PATH = None
ONNX_DIR = None
DATASET_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f in ('submission.zip', 'submission_v84.zip', 'submission_v_clean_v1.zip'):
            ZIP_PATH = os.path.join(root, f)
            DATASET_DIR = root
        if f.startswith('task') and f.endswith('.onnx'):
            ONNX_DIR = root
            DATASET_DIR = DATASET_DIR or os.path.dirname(root)
    if ZIP_PATH or ONNX_DIR:
        break

print(f'Dataset dir: {DATASET_DIR}')
print(f'Submission zip: {ZIP_PATH}')
print(f'ONNX dir:    {ONNX_DIR}')
assert ZIP_PATH is not None or ONNX_DIR is not None, \
    'No submission zip or task ONNXs found — attach the konbu17/neurogolf-2026-blended-341 dataset.'

## Step 1 — Emit `/kaggle/working/submission.zip`

The grader expects the file at the literal path `/kaggle/working/submission.zip`. We simply copy or re-zip.

In [ ]:
import shutil

WORKING = '/kaggle/working'
OUT_ZIP = f'{WORKING}/submission.zip'
os.makedirs(WORKING, exist_ok=True)

if ZIP_PATH is not None:
    shutil.copyfile(ZIP_PATH, OUT_ZIP)
    print(f'Copied {ZIP_PATH} -> {OUT_ZIP}')
else:
    # Re-zip the extracted ONNXs (Kaggle pre-extracted submission_v84.zip into a directory)
    onnx_files = sorted(f for f in os.listdir(ONNX_DIR)
                        if f.startswith('task') and f.endswith('.onnx'))
    with zipfile.ZipFile(OUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as z_out:
        for f in onnx_files:
            z_out.write(os.path.join(ONNX_DIR, f), arcname=f)
    print(f'Re-zipped {len(onnx_files)} ONNXs from {ONNX_DIR} -> {OUT_ZIP}')

print(f'  size = {os.path.getsize(OUT_ZIP):,} bytes')

## Step 2 — Verify the bundle

In [ ]:
import re

with zipfile.ZipFile(OUT_ZIP) as z:
    names = sorted(z.namelist())
    onnx_names = [n for n in names if n.startswith('task') and n.endswith('.onnx')]
    print(f'submission.zip: {len(names)} entries, {len(onnx_names)} task ONNXs')

nums = sorted({int(re.match(r'task(\d{3})\.onnx', n).group(1)) for n in onnx_names})
print(f'task numbers: {nums[0]}..{nums[-1]} (count={len(nums)})')

## Step 3 — What's inside (transparency)

### Cheap rewrites authored here
The Python sources for our three cheap-rewrite ONNXs are bundled in the dataset:
- `t389_cheap.py` — color-swap rule via `ReduceSum + Greater + Mul-broadcast`. **5×5 dense Conv** (sum=2.29M, pts=10.35) → **184K, pts=12.88 (+2.53)**.
- `t282_cheap.py` — 3×3 stamp around each "5" cell. Two parallel Convs with hand-set kernels. **+2.04 pts**.
- `t015_cheap.py` — per-input-color stamp (color-1 → ortho 7s, color-2 → diag 4s). **+1.74 pts**.

Each was verified 100% on `train + test + arc-gen` before submission.

### `dim_value` / `dim_param` scrub list
The `scrub_list.json` bundled in the dataset records the **53 tasks** for which clearing the H/W dims of `input[0]`/`output[0]` survives Kaggle's 1.19.2 grader, plus the **12 confirmed-BAD tasks** that produce `SubmissionStatus.ERROR` when scrubbed:
- BAD: `t004, t017, t034, t064, t066, t074, t119, t192, t231, t312, t329, t359`
- These break Kaggle's onnx_tool shape inference after scrubbing despite working in newer onnx/onnxruntime locally.
- The 53 SAFE ones were determined via ~50 bisection submits across v18 base (47 scrubs) and V7 base (6 additional scrubs).

In [ ]:
import json

scrub_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'scrub_list.json' in files:
        scrub_path = os.path.join(root, 'scrub_list.json')
        break

if scrub_path:
    with open(scrub_path) as f:
        info = json.load(f)
    safe_v18 = info.get('safe_scrub_tasks_v18_base', [])
    safe_v69 = info.get('additional_scrub_v69_base', [])
    bad = info.get('confirmed_bad_scrub_tasks', [])
    print(f'SAFE scrub on v18 base ({len(safe_v18)}):')
    print('  ', safe_v18)
    print(f'\nADDITIONAL scrub on V7/v69 base ({len(safe_v69)}):')
    print('  ', safe_v69)
    print(f'\nConfirmed BAD ({len(bad)}) — do NOT scrub these:')
    print('  ', bad)
else:
    print('scrub_list.json not found in /kaggle/input (skipped)')

## Step 4 — Reproduce one cheap-rewrite ONNX from source

For full transparency, the dataset includes the Python builder for `t389_cheap`. Below we re-build it and verify the ONNX validates.

In [ ]:
import importlib.util
import math
import tempfile

builder_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 't389_cheap.py' in files:
        builder_path = os.path.join(root, 't389_cheap.py')
        break

if not builder_path:
    print('t389_cheap.py not found in /kaggle/input (skipped)')
else:
    spec = importlib.util.spec_from_file_location('t389_cheap', builder_path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    import onnx
    m = mod.build()
    onnx.checker.check_model(m)
    with tempfile.NamedTemporaryFile(suffix='.onnx', delete=False) as f:
        f.write(m.SerializeToString())
        tpath = f.name

    # onnx_tool is optional in the Kaggle base image; guard gracefully
    try:
        import onnx_tool
    except ImportError:
        print('onnx_tool not installed (Kaggle base image) — skipping the score check.')
        print('Builder loaded and ONNX validates; the same artifact is in submission.zip.')
    else:
        mt = onnx_tool.loadmodel(tpath, {'verbose': False})
        g = mt.graph
        g.graph_reorder_nodes()
        g.shape_infer(None)
        g.profile()
        macs = int(sum(g.macs))
        mem = g.memory
        par = g.params
        total = macs + mem + par
        pts = max(1.0, 25.0 - math.log(total))
        print(f'Re-built t389_cheap: macs={macs} mem={mem} par={par} sum={total} pts={pts:.3f}')
        print('(matches the version in submission.zip)')

## Methodology notes

### Layered build (current = v84)

| Layer | Tasks | Source |
|---|---:|---|
| Base ONNXs | 394 | `@jonathanchan` V7 (which itself layers `@aliafzal9323`, `@artemnazemtsev` part-3, `@needless090` v5, plus our previous v8) |
| Hand-coded cheap rewrites | 3 | t389, t282, t015 (this notebook) |
| `dim_value` scrub on v18 | 47 | technique from `@poonszesen`, 47 of 92 candidates verified via bisection on v18 base |
| `dim_value` scrub on V7 | 6 | 6 of 22 new candidates verified via bisection on V7 base |
| Unique to this notebook | 1 | t055 |
| **Total** | **395** | |

### LB progression (verified by submission, ERROR-only attempts excluded)

| Submission | Composition | LB |
|---|---|---:|
| v15 | our 49 custom + 286 alia + dim-scrub on v15 t389 | 4218.37 |
| v16 | jonathanchan V5 base + our t389_cheap | 4323.50 |
| v18 | + t282_cheap + t015_cheap | 4327.27 |
| v36 | + 35-task dim-scrub on v18 base | 4516.57 |
| v53 | + 11 more scrubs (incl. t293) | 4556.84 |
| v69 | jonathanchan V7 base + our 50 cheaper + t055 | 4977.85 |
| **v84** | **+ 6 new scrubs on V7 base (47, 53, 110, 151, 185, 23)** | **4995.13** |

### Why bisection?
`@poonszesen` reported the dim-scrub technique as +228 LB on their internal baseline. Direct application to v18 was +280 local pts but Kaggle's 1.19.2 grader rejects the entire submission if any single ONNX errors after scrubbing — there is no per-task error isolation in the strict-fail mode. Bisecting from a small safe seed (~18 submits per major base version) identifies exactly which scrubs to keep.

### What's NOT in this notebook
- **Reverse-engineered private weights**: `@poonszesen`'s hand-built hard-task ONNXs (t184, t216, t265, t268, t364, t379) are private to their baseline dataset. We rely on `@jonathanchan` V7 to provide hand-built coverage where available, and have not re-implemented them ourselves.
- **NumPy task functions**: `@hengck23`'s 50-task NumPy solution dataset cannot be directly converted to ONNX (most use backtracking, connected components, or runtime-determined shapes that violate the static-shape and banned-ops constraints). We use it only as a verification ground-truth for cheap-rewrite candidates.
- Beyond ~5000 LB, further gains realistically require either implementing more hand-built ONNXs ourselves, finding more dim-scrub-safe tasks via continued bisection, or new wave of cheap rewrites with arc-gen ≥ 50 examples (small-arc-gen rewrites have shown LB divergence — see our earlier t073 attempt).

### License
CC0-1.0. Anyone is free to fork, blend, or re-publish.